# 09. 시장가·즉시체결 3-outcome 재학습

지정가 체결 확률과 `NO_FILL`을 제거한다. 판단 시점 종가를 즉시 시장가 체결 proxy로 사용하고, 이후 10분 동안 `TP +5% / SL -3% / TIMEOUT`만 예측한다. 기존 수수료와 매도 슬리피지 0.1%는 유지한다.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'AGENT.md').exists() and (candidate / 'README.md').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다.')

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.immediate_fill_labeling import create_immediate_artifacts
from src.fixed_train_test import run_fixed_train_test_experiment

assert 'envs/urban' in str(Path(sys.executable).resolve()), sys.executable
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 120)
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'python: {sys.executable}')

PROJECT_ROOT: /home/user/urbandatalab/YSLee/code/Detecting-entry-signals-for-short-term-trades-right-before-a-rapid-price-surge
python: /home/user/anaconda3/envs/urban/bin/python


## 1. 즉시체결 dual-path 라벨 재생성

진입가는 `close_t`이며 체결률은 100%다. 첫 미래 봉 open으로 생기는 gap도 진입가에서 open까지의 가격 구간으로 barrier 판정에 포함한다.

In [2]:
label_result = create_immediate_artifacts(PROJECT_ROOT)
labels = label_result['labels']
display(labels['dual_outcome_10m'].value_counts().rename('samples').to_frame())
display(label_result['summary'].pivot(index='session', columns='dual_outcome_10m', values='samples').fillna(0).astype(int))
assert not labels['dual_outcome_10m'].eq('NO_FILL').any()
assert (labels['entry_price'] - labels['reference_close']).abs().max() == 0
print(f"labels: {len(labels):,}, confirmed: {int(labels['dual_agreement_10m'].sum()):,}, NO_FILL: 0")

,samples
dual_outcome_10m,
TIMEOUT,32312
SL,11247
TP,5455
AMBIGUOUS,70


dual_outcome_10m,AMBIGUOUS,SL,TIMEOUT,TP
session,,,,
session_2026-07-07,0,391,2070,216
session_2026-07-08,1,1820,4367,807
session_2026-07-09,7,1338,4779,582
session_2026-07-10,24,2032,3682,1145
session_2026-07-13,14,1211,4106,641
session_2026-07-14,8,1220,2961,466
session_2026-07-15,15,1745,4043,840
session_2026-07-16,0,983,2886,517
session_2026-07-17,1,507,3418,241


labels: 49,084, confirmed: 49,014, NO_FILL: 0


## 2. Validation 없는 시간순 7 Train / 2 Test 학습

Reduced V2의 90개 feature와 hidden 32 구조를 유지한다. 10분 event sampling, 날짜 균등 loss, 15 epoch 고정 학습을 사용하며 출력은 `TP / SL / TIMEOUT` 세 개다.

In [3]:
CONFIG_PATH = PROJECT_ROOT / 'configs/fixed_train_test_immediate_fill.yaml'
result = run_fixed_train_test_experiment(PROJECT_ROOT, CONFIG_PATH)
print(f"device: {result['device']}")
print(f"features: {result['feature_count']}, parameters: {result['parameter_count']:,}")
display(result['sampling_balance'])

device: cuda
features: 90, parameters: 3,011


,session,source_rows,is_train,sampled_rows,total_loss_weight
0,session_2026-07-07,2677,True,282,603.714294
1,session_2026-07-08,6994,True,735,603.714294
2,session_2026-07-09,6699,True,695,603.714294
3,session_2026-07-10,6859,True,718,603.714294
4,session_2026-07-13,5958,True,616,603.714294
5,session_2026-07-14,4647,True,489,603.714294
6,session_2026-07-15,6628,True,691,603.714294
7,session_2026-07-16,4386,False,0,0.000000
8,session_2026-07-17,4166,False,0,0.000000


## 3. Train–Test 일반화 지표

In [4]:
display(result['metrics'])
display(result['generalization_gap'])

,evaluation_group,session,samples,multiclass_logloss,multiclass_brier,accuracy,macro_pr_auc,tp_pr_auc,tp_roc_auc,tp_positive_rate,tp_ece,return_spearman,return_mae,top1_mean_actual_return,top5_mean_actual_return
0,train,ALL,40462,0.689880,0.397623,0.710716,0.554097,0.255141,0.756534,0.116084,0.006470,0.143321,0.019285,-0.000329,-0.001938
1,test,ALL,8552,0.576087,0.322372,0.778882,0.543014,0.200500,0.763667,0.088634,0.010259,0.148424,0.016094,-0.001334,-0.002565
2,train,session_2026-07-07,2677,0.499349,0.277069,0.809862,0.563833,0.161496,0.774482,0.080687,0.020882,0.214740,0.013896,0.000932,-0.003111
3,train,session_2026-07-08,6994,0.759948,0.445701,0.665856,0.514212,0.218227,0.712344,0.115385,0.004351,0.148251,0.019965,-0.003844,-0.002762
4,train,session_2026-07-09,6699,0.640042,0.358916,0.762353,0.531468,0.240246,0.758451,0.086879,0.011819,0.102593,0.017588,0.003099,-0.002844
5,train,session_2026-07-10,6859,0.754735,0.443108,0.655927,0.578326,0.301109,0.736777,0.166934,0.026176,0.144316,0.022162,-0.000352,-0.002641
6,train,session_2026-07-13,5958,0.649692,0.368122,0.743370,0.553484,0.276653,0.770956,0.107586,0.010501,0.094781,0.018834,0.001124,-0.001263
7,train,session_2026-07-14,4647,0.681296,0.391927,0.713148,0.559502,0.247676,0.780057,0.100280,0.023061,0.147346,0.019000,0.000711,-0.001741
8,train,session_2026-07-15,6628,0.718295,0.418144,0.691460,0.558500,0.248245,0.729931,0.126735,0.006411,0.157870,0.020087,-0.005294,-0.000648
9,test,session_2026-07-16,4386,0.670833,0.386166,0.719106,0.549165,0.213093,0.717574,0.117875,0.009949,0.159172,0.018213,-0.006952,-0.004316


,metric,train,test,test_minus_train
0,multiclass_logloss,0.689880,0.576087,-0.113793
1,macro_pr_auc,0.554097,0.543014,-0.011083
2,tp_pr_auc,0.255141,0.200500,-0.054640
3,tp_roc_auc,0.756534,0.763667,0.007133
4,return_spearman,0.143321,0.148424,0.005103
5,return_mae,0.019285,0.016094,-0.003191
6,top1_mean_actual_return,-0.000329,-0.001334,-0.001005
7,top5_mean_actual_return,-0.001938,-0.002565,-0.000627


## 4. 즉시체결 순차 백테스트

Train outcome으로 threshold를 최적화하지 않고 Train 예측 점수 99% 분위수를 고정한다. 신호가 발생하면 모두 즉시체결되지만, 동일 종목의 기존 포지션이 열려 있으면 중복 주문은 건너뛴다.

In [5]:
display(result['threshold'])
display(result['backtest_metrics'])
display(result['session_metrics'])

,threshold,method,train_score_quantile,uses_outcome_for_threshold_selection,temperature,validation_used
0,0.000546,train_score_quantile_without_outcome_tuning,0.99,False,1.0,False


,signals_above_threshold,ten_minute_signal_clusters,order_attempts,filled_trades,tp_trades,sl_trades,timeout_trades,tp_precision_given_fill,mean_net_return_per_fill,median_net_return_per_fill,net_return_on_deployed_capital,return_on_initial_capital,total_net_pnl,profit_factor,max_drawdown,skipped_same_symbol,skipped_position_limit,skipped_cash_limit,threshold,ending_cash,evaluation_group
0,405,173,304,304,106,153,45,0.348684,0.000518,-0.032939,0.000524,0.015925,159.253571,1.030165,-0.045546,101,0,0,0.000546,10159.253571,train
1,25,15,21,21,4,12,5,0.190476,-0.008854,-0.032958,-0.008862,-0.018611,-186.108107,0.536510,-0.027613,4,0,0,0.000546,9813.891893,test


,signals_above_threshold,ten_minute_signal_clusters,order_attempts,filled_trades,tp_trades,sl_trades,timeout_trades,tp_precision_given_fill,mean_net_return_per_fill,median_net_return_per_fill,net_return_on_deployed_capital,return_on_initial_capital,total_net_pnl,profit_factor,max_drawdown,skipped_same_symbol,skipped_position_limit,skipped_cash_limit,threshold,ending_cash,evaluation_group,session
0,3,2,2,2,1,1,0,0.500000,0.006903,0.006903,0.007010,0.001397,13.967950,1.426760,-0.003273,1,0,0,0.000546,10013.967950,train,session_2026-07-07
1,32,19,28,28,3,15,10,0.107143,-0.008488,-0.032990,-0.008501,-0.023795,-237.945427,0.555620,-0.039261,4,0,0,0.000546,9762.054573,train,session_2026-07-08
2,21,16,19,19,9,6,4,0.473684,0.009353,0.008446,0.009334,0.017728,177.284866,1.728141,-0.019541,2,0,0,0.000546,10177.284866,train,session_2026-07-09
3,132,53,102,102,37,53,12,0.362745,-0.000357,-0.032944,-0.000353,-0.003599,-35.991079,0.980250,-0.045336,30,0,0,0.000546,9964.008921,train,session_2026-07-10
4,55,23,43,43,18,19,6,0.418605,0.005358,-0.003032,0.005383,0.023105,231.045351,1.344612,-0.019605,12,0,0,0.000546,10231.045351,train,session_2026-07-13
5,65,20,45,45,17,25,3,0.377778,-0.000394,-0.032950,-0.000380,-0.001710,-17.095380,0.979506,-0.030573,20,0,0,0.000546,9982.904620,train,session_2026-07-14
6,97,40,65,65,21,34,10,0.323077,0.000420,-0.032969,0.000431,0.002799,27.987289,1.024533,-0.021256,32,0,0,0.000546,10027.987289,train,session_2026-07-15
7,14,8,12,12,2,9,1,0.166667,-0.017504,-0.033067,-0.017509,-0.021020,-210.197107,0.305060,-0.021020,2,0,0,0.000546,9789.802893,test,session_2026-07-16
8,11,7,9,9,2,3,4,0.222222,0.002679,0.004917,0.002678,0.002409,24.088999,1.243156,-0.006593,2,0,0,0.000546,10024.088999,test,session_2026-07-17


## 5. 결론과 제한

7/16~17은 이전 실험에서 이미 확인한 비독립 test다. 이 결과는 지정가 체결 가정 제거의 영향을 확인하기 위한 것이며 배포 승인에 사용하지 않는다.

In [6]:
test_row = result['backtest_metrics'].query("evaluation_group == 'test'").iloc[0]
display(Markdown(
    f"**Test 신호 {int(test_row['signals_above_threshold'])}개, 즉시체결 거래 {int(test_row['filled_trades'])}건, "
    f"평균 순수익률 {test_row['mean_net_return_per_fill']:.3%}.**"
))

**Test 신호 25개, 즉시체결 거래 21건, 평균 순수익률 -0.885%.**

## 6. 저장 artifact

In [7]:
paths = {**label_result['paths'], **result['paths']}
display(pd.DataFrame({'artifact': paths.keys(), 'path': map(str, paths.values())}))

,artifact,path
0,labels,/home/user/urbandatalab/YSLee/data/stock_data/...
1,summary,/home/user/urbandatalab/YSLee/data/stock_data/...
2,tabular,/home/user/urbandatalab/YSLee/data/stock_data/...
3,schema,/home/user/urbandatalab/YSLee/data/stock_data/...
4,predictions,/home/user/urbandatalab/YSLee/data/stock_data/...
5,metrics,/home/user/urbandatalab/YSLee/data/stock_data/...
6,generalization_gap,/home/user/urbandatalab/YSLee/data/stock_data/...
7,training_history,/home/user/urbandatalab/YSLee/data/stock_data/...
8,sampling_balance,/home/user/urbandatalab/YSLee/data/stock_data/...
9,payoffs,/home/user/urbandatalab/YSLee/data/stock_data/...
